**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Diffusion II: Score Matching & SDEs

The rigorous sequel [Diffusion Models](./Diffusion_Models.ipynb) gestured at: what the network *actually* learns is the **score** $\nabla_x \log p(x)$, the discrete chain is an [SDE](../Intro_Math/Stochastic_Processes/Stochastic_Processes_2.ipynb) in disguise, and generation is that SDE run backwards. Verified on a Gaussian mixture where the true score is available in closed form — the oracle most tutorials never check.

## 1. Pre-requisites

[Diffusion Models](./Diffusion_Models.ipynb) (the practical loop), [Stochastic Processes II](../Intro_Math/Stochastic_Processes/Stochastic_Processes_2.ipynb) S4 (Brownian motion, Itô).

In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

# ground-truth distribution: a 2-D Gaussian mixture — score known in CLOSED FORM
centers = torch.tensor([[-2.0, 0.0], [2.0, 0.0], [0.0, 2.2]])
sig2 = 0.35**2
def sample_data(n):
    idx = torch.randint(0, 3, (n,))
    return centers[idx] + 0.35*torch.randn(n, 2)
def true_score(x, t_var=0.0):
    """∇ log p_t for the mixture convolved with N(0, t_var) — exact."""
    v = sig2 + t_var
    d2 = ((x[:, None, :] - centers[None])**2).sum(-1)
    w = torch.softmax(-d2/(2*v), dim=1)
    mu_post = (w[:, :, None] * centers[None]).sum(1)
    return (mu_post - x) / v

---
### 🕐 Session 1 of 3 — *The Score Function* (~40 min)
**Goal:** learn ∇ log p by denoising; verify against the closed-form mixture score.
**Builds on:** [Diffusion Models](./Diffusion_Models.ipynb). &nbsp; **Feeds into:** Session 2 (Langevin & SDEs).

---

## 2. The Gradient of the Log-Density

💡 **Intuition.** The score $s(x) = \nabla_x \log p(x)$ is a *compass field*: at every point it points toward higher probability. You never need the (intractable) normalizing constant — gradients of $\log p$ kill it. And the miracle that makes it learnable: **denoising score matching** — training a network to predict the noise added to data is, up to scale, training it to output the score of the *noised* distribution ($s = -\varepsilon/\sigma$). Diffusion I's 'predict the noise' loss was secretly score estimation all along. Here we can *prove* it: our mixture's score has a closed form to compare against.

In [2]:
# train a denoiser at ONE noise level; compare its implied score with the exact one
sigma_noise = 0.5
net = nn.Sequential(nn.Linear(2, 128), nn.SiLU(), nn.Linear(128, 128), nn.SiLU(), nn.Linear(128, 2))
opt = torch.optim.Adam(net.parameters(), lr=2e-3)
for step in range(3000):
    x0 = sample_data(512)
    eps = torch.randn_like(x0)
    xt = x0 + sigma_noise*eps
    loss = ((net(xt) - eps)**2).mean()
    opt.zero_grad(); loss.backward(); opt.step()

# ORACLE: implied score −net(x)/σ vs the closed-form mixture score at this noise level
g = torch.linspace(-3.5, 3.5, 24)
GX, GY = torch.meshgrid(g, g, indexing="xy")
pts = torch.stack([GX.ravel(), GY.ravel()], 1)
with torch.no_grad():
    s_learned = -net(pts)/sigma_noise
s_exact = true_score(pts, t_var=sigma_noise**2)
cos = nn.functional.cosine_similarity(s_learned, s_exact, dim=1)
rel = (s_learned - s_exact).norm(dim=1) / (s_exact.norm(dim=1) + 1e-6)
print(f"learned vs exact score: median cosine {cos.median():.4f}, median relative error {rel.median():.3f}")

plt.figure(figsize=(5.2, 4.6))
plt.quiver(GX, GY, s_exact[:,0].reshape(24,24), s_exact[:,1].reshape(24,24), color="gray", alpha=0.6, label="exact")
plt.quiver(GX, GY, s_learned[:,0].reshape(24,24), s_learned[:,1].reshape(24,24), color="crimson", alpha=0.6, scale=None, label="learned")
plt.scatter(*sample_data(400).T, s=2, alpha=0.3)
plt.legend(fontsize=7); plt.title("the compass field, learned from denoising alone")
plt.tight_layout(); plt.show()

learned vs exact score: median cosine 0.9987, median relative error 0.098


/tmp/ipykernel_2995578/43406241.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 3 — *Langevin Dynamics & the Forward SDE* (~40 min)
**Goal:** climb the score with noise: Langevin sampling; the diffusion chain as an SDE.
**Builds on:** Session 1; [Stochastic Processes II](../Intro_Math/Stochastic_Processes/Stochastic_Processes_2.ipynb). &nbsp; **Feeds into:** Session 3 (the reverse SDE & probability flow).

---

## 3. Sampling = Noisy Gradient Ascent

💡 **Intuition.** Given a score, **Langevin dynamics** samples: $x \mathrel{+}= \frac{\eta}{2} s(x) + \sqrt{\eta}\, \xi$ — climb the compass field, but inject just enough noise that you *explore* the distribution instead of collapsing to its modes ([SGD's noise](../Intro_Math/Optimization/Optimization.ipynb), now load-bearing). The catch that motivated diffusion: with far-apart modes, plain Langevin mixes badly — which is why diffusion runs a *family* of scores across noise levels: high noise merges the modes for easy travel, low noise sharpens the details. And in the continuum, Diffusion I's chain **is** the SDE $dx = -\tfrac12\beta x\, dt + \sqrt{\beta}\, dB$ — an Ornstein–Uhlenbeck process whose marginals we can check exactly.

In [3]:
# ORACLE: the forward SDE's variance must follow the OU closed form
beta = 1.2
dt = 0.001
T_steps = 2000
x = sample_data(4000)
var_path = []
for k in range(T_steps):
    x = x - 0.5*beta*x*dt + np.sqrt(beta*dt)*torch.randn_like(x)
    if k % 100 == 0: var_path.append(x.var().item())
t_ax = np.arange(0, T_steps, 100)*dt
var_theory = np.exp(-beta*t_ax)*sample_data(20000).var().item() + (1 - np.exp(-beta*t_ax))
plt.figure(figsize=(7, 2.4))
plt.plot(t_ax, var_path, "o", markersize=4, label="simulated variance")
plt.plot(t_ax, var_theory, "k-", linewidth=1, label="OU closed form")
plt.legend(fontsize=8); plt.xlabel("t"); plt.title("the forward SDE, audited against Itô's answer")
plt.tight_layout(); plt.show()
print(f"max |simulated − OU theory| variance: {np.abs(np.array(var_path)-var_theory).max():.3f}")

max |simulated − OU theory| variance: 0.030


/tmp/ipykernel_2995578/547938826.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 3 of 3 — *The Reverse SDE & Probability Flow* (~40 min)
**Goal:** run time backwards with the score; sample the mixture and audit mode weights.
**Builds on:** Session 2.

---

## 4. Anderson's Time Machine

💡 **Intuition.** The stunning theorem (Anderson, 1982): the forward SDE has an exact **reverse**: $dx = [-\tfrac12\beta x - \beta\, s_t(x)]\, dt + \sqrt{\beta}\, d\bar{B}$ — identical dynamics plus a score-guided drift. Everything unknown about 'undoing noise' is packed into $s_t$, the exact object Session 1 taught us to learn. Drop the noise term and halve the score drift and you get the **probability-flow ODE** — deterministic, same marginals, the bridge to flow matching and fast samplers (DDIM is its discretization).

In [4]:
# train a time-conditional score net, then sample by reverse SDE — audit the result
class ScoreNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = nn.Sequential(nn.Linear(2+8, 128), nn.SiLU(), nn.Linear(128, 128), nn.SiLU(), nn.Linear(128, 2))
    def forward(self, x, tt):
        emb = torch.cat([torch.sin(tt[:, None]*torch.arange(1, 5)), torch.cos(tt[:, None]*torch.arange(1, 5))], 1)
        return self.f(torch.cat([x, emb], 1))

T_max = 2.0
score_net = ScoreNet()
opt = torch.optim.Adam(score_net.parameters(), lr=2e-3)
for step in range(6000):
    x0 = sample_data(512)
    tt = torch.rand(512)*T_max + 1e-3
    a = torch.exp(-0.5*beta*tt)[:, None]                       # OU mean decay
    var = (1 - torch.exp(-beta*tt))[:, None]
    eps = torch.randn_like(x0)
    xt = a*x0 + var.sqrt()*eps
    target = -eps/var.sqrt()                                   # the score of the perturbed marginal
    loss = ((score_net(xt, tt) - target)**2 * var).mean()      # standard weighting
    opt.zero_grad(); loss.backward(); opt.step()

# reverse SDE from pure noise
x = torch.randn(6000, 2)
n_rev = 400
dt_r = T_max/n_rev
with torch.no_grad():
    for k in range(n_rev):
        tt = torch.full((len(x),), T_max - k*dt_r)
        s = score_net(x, tt)
        x = x + (0.5*beta*x + beta*s)*dt_r + np.sqrt(beta*dt_r)*torch.randn_like(x)

plt.figure(figsize=(4.6, 4))
plt.scatter(*x.T, s=2, alpha=0.2, label="reverse-SDE samples")
plt.scatter(*centers.T, marker="*", s=150, c="crimson", label="true mode centers")
plt.legend(fontsize=7); plt.axis("equal"); plt.title("noise → mixture, via the learned score")
plt.tight_layout(); plt.show()

# ORACLE: mode weights should be ≈ 1/3 each; mode means ≈ the centers
assign = ((x[:, None, :] - centers[None])**2).sum(-1).argmin(1)
for k in range(3):
    frac = (assign == k).float().mean()
    mu = x[assign == k].mean(0)
    print(f"mode {k}: weight {frac:.3f} (true 0.333)   center {mu.numpy().round(2)} (true {centers[k].numpy()})")

mode 0: weight 0.354 (true 0.333)   center [-2.08  0.05] (true [-2.  0.])
mode 1: weight 0.330 (true 0.333)   center [ 2.06 -0.02] (true [2. 0.])
mode 2: weight 0.316 (true 0.333)   center [0.03 2.19] (true [0.  2.2])


/tmp/ipykernel_2995578/2519242878.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 5. Conclusion

'Predict the noise' is score estimation (verified against a closed-form score, cosine ≈ 1); the chain is an OU SDE (variance audited against Itô); and Anderson's reverse SDE turns the learned compass into a sampler whose mode weights and centers match the truth. Diffusion I's recipe now has its complete mathematical spine.

---
## Where next

- [Stochastic Processes II](../Intro_Math/Stochastic_Processes/Stochastic_Processes_2.ipynb) — the Itô calculus underneath.
- [Optimal Transport](./Optimal_Transport.ipynb) — probability flow as a transport map; flow matching lives here.